# 多轮对话与调试

显式管理历史，并用本地调用记录查看输入、输出与耗时。

按章节从上到下运行。说明、代码、输出应一起阅读。所有模型调用都访问真实服务并可能计费，重新运行也会产生请求。示例制度和数字仅用于课堂练习。


In [11]:
import json
from pathlib import Path
from course_api import pretty, answer, text, chat, vision, image_part, config_status
from IPython.display import display, Image, Markdown
pretty(config_status())


{
  "Python配置": "请同时检查 sys.executable",
  "对话模型": "deepseek-flash",
  "对话地址": "https://api.deepseek.com",
  "对话密钥已配置": true
}


## 1. 历史列表

HTTP请求之间不会自动共享本地变量。每次发送需要的历史消息，才能完成追问。


In [12]:
history = [{'role':'system','content':'你是Python学习助手，回答简短，不编造学生信息。'}]
def ask_followup(question):
    user = {'role':'user','content':question}
    reply = text(history + [user])
    history.extend([user, {'role':'assistant','content':reply}])
    return reply
answer(ask_followup('我刚学Python，解释一下列表和元组。'))


列表和元组都是有序序列，主要区别：

- **列表 list**：用 `[]`，可变，可增删改。如 `a = [1, 2, 3]`，`a.append(4)`。
- **元组 tuple**：用 `()`，不可变，创建后不能修改。如 `t = (1, 2, 3)`。

选择：需要修改用列表，数据固定用元组（更安全、可作字典键）。

### 运行后观察

下一格再次调用时，history会包含上一轮的用户问题与助手回答。


## 2. 完整回合截断

只保留最近完整回合，保留system消息。这个简化方法适用于本例严格用户/助手交替的列表。


In [13]:
answer(ask_followup('给我一个关于它们区别的短练习。'))
recent = [history[0]] + history[1:][-4:]
pretty([{'role':m['role'],'characters':len(m['content'])} for m in recent])


**练习：**

```python
a = [1, 2, 3]
t = (1, 2, 3)

a[0] = 10   # ①
t[0] = 10   # ②
```

1. ① 和 ② 分别会发生什么？为什么？
2. 再写一行代码，给 `a` 末尾加一个元素 `4`，并打印 `a`。

先自己试，写完可以发给我看。

[
  {
    "role": "system",
    "characters": 26
  },
  {
    "role": "user",
    "characters": 20
  },
  {
    "role": "assistant",
    "characters": 165
  },
  {
    "role": "user",
    "characters": 15
  },
  {
    "role": "assistant",
    "characters": 159
  }
]


### 运行后观察

此处保留最近两轮。工具消息不能这样随意截断，应保留工具调用和返回配对。字符数也不等于Token数。


## 3. 本地调用记录

记录本次实际发送的消息、返回正文、耗时和服务返回的Token用量。`chat`返回结构中`message`保存模型消息，`usage`保存用量，`seconds`保存耗时。记录仅保留本次课堂样例，不保存请求头或密钥。


In [14]:
sample_messages = [{'role':'user','content':'用一句话解释什么是上下文。'}]
response = chat(sample_messages)
record = {
    'input': sample_messages,
    'output': response['message'].get('content'),
    'seconds': response['seconds'],
    'usage': response['usage'],
}
pretty(record)
answer(record['output'])


{
  "input": [
    {
      "role": "user",
      "content": "用一句话解释什么是上下文。"
    }
  ],
  "output": "上下文就是你在说话或做事时，周围那些能让对方准确理解你意思的所有背景信息。",
  "seconds": 1.07,
  "usage": {
    "prompt_tokens": 10,
    "completion_tokens": 20,
    "total_tokens": 30,
    "prompt_tokens_details": {
      "cached_tokens": 0
    },
    "prompt_cache_hit_tokens": 0,
    "prompt_cache_miss_tokens": 10
  }
}


上下文就是你在说话或做事时，周围那些能让对方准确理解你意思的所有背景信息。

### 运行后观察

先核对input是否完整，再检查output是否回答了问题。seconds反映本次请求的等待耗时，usage是服务返回的用量信息。格式正确和请求成功都不能代替内容核对。不要在演示中输出Authorization请求头或.env文件。


## 4. 排错顺序

先看输入，再看消息角色、模型响应，最后看解析与界面显示。记录一次失败应包含触发条件和预期结果。


In [15]:
pretty({'messages':len(history), 'roles':[m['role'] for m in history],
        'last_user':next(m['content'] for m in reversed(history) if m['role']=='user')})


{
  "messages": 5,
  "roles": [
    "system",
    "user",
    "assistant",
    "user",
    "assistant"
  ],
  "last_user": "给我一个关于它们区别的短练习。"
}


### 运行后观察

查看是否漏发历史、是否选错内核、是否因为展示截断误以为没有第二份回答。不要打印.env和请求Authorization头。


## 5. 不带历史与带历史对照

先记录一个具体学习背景。两份请求使用相同追问，A只发追问，B带上真实前一轮回答，分别显示。


In [16]:
system_msg = {'role':'system','content':'你是Python学习助手，回答简短。用户提供的学习背景应保持一致。'}
first_user = {'role':'user','content':'我刚学Python，只学过变量和循环。请用我学过的内容解释列表。'}
first_answer = text([system_msg,first_user])
full_history = [first_user,{'role':'assistant','content':first_answer}]
follow_question = '按我刚才的基础出一道自测题，先不要给答案。'
reply_without = text([system_msg,{'role':'user','content':follow_question}])
answer(reply_without)


好的，请先告诉我你刚才的学习背景（学到哪、掌握哪些内容），我才能按你的基础出题。

### 运行后观察

A没有收到“只学过变量和循环”的背景。它有可能给出碰巧合适的题，也可能使用未学概念；不能因一次碰巧正确就认为服务自动保存了本地会话。


## 6. 带历史的请求

B显式发送前一轮背景和回答。对照同一个标准检查题目难度、是否提前泄露答案，以及是否沿用用户背景。


In [17]:
reply_with = text([system_msg]+full_history+[{'role':'user','content':follow_question}])
answer(reply_with)


题目：

```python
nums = [3, 8, 5, 2]
```

请用你学过的循环，算出这个列表里所有数字的和，并打印结果。

提示：先想一个变量来存“和”，初始值该是多少？

### 运行后观察

检查本次实际消息，而不是把模型自称“记得”当作状态证据。A/B分格显示，避免输出截断。


## 7. 回合窗口与背景丢失

下面从真实对话历史中截取最近完整回合。摘要是人工选出的已知事实，专门用来观察状态设计，不是假造API回复。


In [18]:
def keep_turns(system, turns, count=1):
    if count < 1 or len(turns) % 2:
        raise ValueError('需要完整用户/助手回合，count至少为1')
    expected = ['user','assistant'] * (len(turns)//2)
    if [m['role'] for m in turns] != expected:
        raise ValueError('本函数只处理严格交替的用户/助手历史')
    return [system] + turns[-2*count:]
turns = full_history + [{'role':'user','content':follow_question},{'role':'assistant','content':reply_with}]
short_messages = keep_turns(system_msg,turns,count=1)
pretty([{'role':m['role'],'content':m['content']} for m in short_messages])
known_context = '已知用户背景：刚学Python，只学过变量和循环。'
summary_messages = [system_msg,{'role':'user','content':known_context}] + turns[-2:]
pretty({'short_count':len(short_messages),'with_background_count':len(summary_messages)})


[
  {
    "role": "system",
    "content": "你是Python学习助手，回答简短。用户提供的学习背景应保持一致。"
  },
  {
    "role": "user",
    "content": "按我刚才的基础出一道自测题，先不要给答案。"
  },
  {
    "role": "assistant",
    "content": "题目：\n\n```python\nnums = [3, 8, 5, 2]\n```\n\n请用你学过的循环，算出这个列表里所有数字的和，并打印结果。\n\n提示：先想一个变量来存“和”，初始值该是多少？"
  }
]
{
  "short_count": 3,
  "with_background_count": 4
}


### 运行后观察

最近一轮的用户消息可能只说“按我刚才的基础”，完整回合虽然保留了，但最初背景仍可能丢失。摘要选择会影响保留内容；生产场景还需更新、纠错与过期策略。


## 8. 会话隔离与成功后更新

为每个会话单独维护列表。先获得真实回复，再一次性追加完整回合，避免请求失败后留下半个回合。此处封装保持最小，便于检查状态。


In [19]:
class StudySession:
    def __init__(self):
        self.turns = []
    def reset(self):
        self.turns.clear()
    def ask(self, question):
        if not question.strip():
            raise ValueError('问题不能为空')
        user = {'role':'user','content':question}
        reply = text([system_msg]+self.turns[-8:]+[user])
        self.turns.extend([user,{'role':'assistant','content':reply}])
        return reply
session_a, session_b = StudySession(), StudySession()
assert session_a.turns is not session_b.turns
session_a.reset()
pretty({'A_history':session_a.turns,'B_history':session_b.turns})


{
  "A_history": [],
  "B_history": []
}


### 运行后观察

本格只检查对象状态，不调用ask就不会产生API请求。后续调用时，两个对象应保持独立，reset清空的是实际列表。


## 9. 保存与比较本地记录

保存本次运行的输入、输出、用量和耗时。记录来自前面真实请求，保存在本机data目录。


In [20]:
Path('data').mkdir(exist_ok=True)
Path('data/本地调用记录.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
pretty({'saved_to':'data/本地调用记录.json','seconds':record['seconds'],'usage':record['usage']})


{
  "saved_to": "data/本地调用记录.json",
  "seconds": 1.07,
  "usage": {
    "prompt_tokens": 10,
    "completion_tokens": 20,
    "total_tokens": 30,
    "prompt_tokens_details": {
      "cached_tokens": 0
    },
    "prompt_cache_hit_tokens": 0,
    "prompt_cache_miss_tokens": 10
  }
}


### 运行后观察

只保存当前样例所需信息，不保存Authorization头和密钥。不同问题的耗时不能直接作为模型快慢比较，应控制输入条件。字符数也不能替代服务返回的Token统计。
